# Roanoke, VA — Land Value Tax Model

**Model type**: Split-rate 4:1 (land taxed at 4× the improvement rate)  
**Levy scope**: City levy only (Roanoke is an independent Virginia city; $1.22/$100 covers all city operations including schools)  
**Tax rate**: $1.22 per $100 of assessed value = 12.2 mills (unchanged since FY2016)  
**Assessment basis**: 100% market value (Virginia Code § 58.1-3201; no assessment ratio)  
**Legal status**: Authorized by Va. Code §58.1-3221.1 since 2002 (original enactment)  
**Data source**: City of Roanoke Parcels FeatureServer (gis03.roanokeva.gov)  
**Revenue target**: ~$95M FY2025 (per news reports: real estate tax projected at $94.7M FY2025, rising to $99.8–101.9M FY2026)  
**Revenue gap note**: Model generates ~$155M (vs. ~$95M official) because Roanoke's public GIS does not encode all local tax-exemption designations. Large institutional non-profits (Carilion Clinic health system, colleges, etc.) appear as taxable `400-Commercial/Industrial` parcels but are exempt via local ordinance under Va. Code §58.1-3607. The official assessment roll (held by the Commissioner of the Revenue) would be required for a complete exemption list. The model is conservative — it overstates the taxable base.

## Column Mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `LANDVAL1` | Current land value at 100% FMV |
| Improvement value | `DWELLINGVA` | Dwelling/improvement value; 0 = vacant |
| Total value | `TOTALVAL1` | Land + improvement |
| Classification | `PROPERTYDE` | Property description encoding type AND exemption reason |
| Parcel ID | `TAXID` | Tax parcel ID |

**Exemption encoding:** `PROPERTYDE` suffix encodes exempt type directly (e.g., `451-Comm/Indust-City` = city-owned commercial, `455-Comm/Indust-Religious` = religious commercial). Taxable parcels have suffixes like `200-SingleFamily`, `400-Commercial/Industrial`. Exemption detection uses PROPERTYDE suffix — but this misses institutional non-profits designated exempt via local ordinance.  
**Assessment ratio**: None (100% FMV per Virginia law)  
**Millage source**: Adopted city rate $1.22/$100 = 12.2 mills; last changed FY2016  
**SCLT note**: Roanoke has a community land trust (SCLT parcels) treated as Exempt here.

## Section 1 — Imports and Setup

In [ ]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'roanoke'
STATE_FIPS = '51'
COUNTY_FIPS = '770'   # Roanoke independent city FIPS
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

# Roanoke adopted real estate tax rate: $1.22 per $100 assessed value (unchanged since FY2016)
# Virginia assesses at 100% market value — no assessment ratio
MILLAGE = 12.2  # mills per $1,000 AV  ($1.22 per $100)
# Official revenue: ~$94.7M FY2025, projected $99.8-101.9M FY2026 (news reports)
# Model generates ~$155M because the public GIS omits institutional non-profit exemptions
# (Carilion Clinic, colleges, etc. exempt under Va. Code §58.1-3607 but labeled as taxable in GIS)
OFFICIAL_REVENUE = 95_000_000  # FY2025 official estimate from news reports

print(f"City: {CITY_NAME}")
print(f"FIPS: {STATE_FIPS}{COUNTY_FIPS}")
print(f"Millage: {MILLAGE} mills  (${MILLAGE/10:.2f} per $100 AV)")

## Section 2 — Fetch / Load Parcel Data

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    # City of Roanoke Parcels FeatureServer
    # get_feature_data_with_geometry builds: {base_url}/{dataset_name}/FeatureServer/{layer_id}/query
    gdf = get_feature_data_with_geometry(
        dataset_name='Parcels',
        base_url='https://gis03.roanokeva.gov/arcgis/rest/services/Public',
        layer_id=0,
        paginate=True,
    )
    # Roanoke is an independent city — all parcels in this service are within the city
    gdf.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf):,} parcels")

print(f"Columns: {list(gdf.columns)}")
print(f"CRS: {gdf.crs}")

Layer metadata CRS WKID: 3857


Total records in Parcels: 44441


Query response spatialReference: {'wkid': 4326, 'latestWkid': 4326}
Fetched records 0 to 1000


Fetched records 1000 to 2000


Fetched records 2000 to 3000


...

Fetched records 42000 to 43000


Fetched records 43000 to 44000


Fetched records 44000 to 44441


Total bounds: [-80.037477  37.211899 -79.87855   37.336382]


Downloaded and cached 44,441 parcels
Columns: ['OBJECTID', 'TAXID', 'GISOBJID', 'LRSN', 'LOCADDR', 'ADDRNBR', 'ADDRSTREET', 'ADDRQUAD', 'ADDRSUFFIX', 'OWNER', 'OWNERADDR1', 'MAILCITY', 'MAILSTATE', 'LEGALACRES', 'ZONEDESC', 'LEGALDESC', 'LANDVAL1', 'LANDVAL2', 'TOTALVAL1', 'TOTALVAL2', 'SALEDATE1', 'SALEDATE2', 'SALEAMT1', 'SALEAMT2', 'GRANTOR1', 'GRANTOR2', 'DOCNUM1', 'DOC2NUM', 'SQFT', 'ACRES', 'TOPODESC1', 'ASSOCNAME', 'OWNER2', 'Shape__Area', 'Shape__Length', 'NEIGHBORHO', 'MAINZIPCOD', 'PROPERTYDE', 'ASSESSMENT', 'ASSESSME_1', 'CHANGEDESC', 'CHANGEDE_1', 'DWELLINGVA', 'DWELLING_1', 'LOCZIP', 'geometry']
CRS: EPSG:4326


In [3]:
# Value validation
print("Value column statistics:")
print(gdf[['LANDVAL1', 'DWELLINGVA', 'TOTALVAL1']].describe())
print(f"\nParcels with $0 total value: {(gdf['TOTALVAL1'].fillna(0)==0).sum():,}")
print(f"Parcels with $0 land value:  {(gdf['LANDVAL1'].fillna(0)==0).sum():,}")
print(f"Parcels with $0 improvement: {(gdf['DWELLINGVA'].fillna(0)==0).sum():,}")
print(f"\nPROPERTYDE distribution (top 20):")
print(gdf['PROPERTYDE'].value_counts().head(20))

Value column statistics:
           LANDVAL1    DWELLINGVA     TOTALVAL1
count  4.443600e+04  4.443600e+04  4.443600e+04
mean   6.522832e+04  2.688983e+05  3.341266e+05
std    3.092060e+05  2.710136e+06  2.844487e+06
min    0.000000e+00  0.000000e+00  0.000000e+00
25%    2.360000e+04  7.040000e+04  1.012000e+05
50%    3.460000e+04  1.478000e+05  1.838000e+05
75%    5.070000e+04  2.160000e+05  2.662000e+05
max    4.375600e+07  4.761611e+08  4.838303e+08

Parcels with $0 total value: 263
Parcels with $0 land value:  263
Parcels with $0 improvement: 8,745

PROPERTYDE distribution (top 20):
PROPERTYDE
200-SingleFamily             29616
100-Vacant Land               5309
400-Commercial/Industrial     2565
140-Commercial Vacant         1841
300-Multifamily               1727
Multi-Family Duplex            961
151-Vacant-SFR-City            686
155-Vacant-Religious           256
455-Comm/Indust-Religious      224
451-Comm/Indust-City           139
154-Vacant-Regional            131
152-Vacant

## Section 3 — Classify and Validate

In [4]:
# Roanoke's PROPERTYDE field encodes both property type AND exemption reason in a single string.
# Exempt suffixes: City, Religious, State, Regional, Charitable, Federal, SCC, Educational, SCLT, Other
# Condo parent parcels (2260-Res Condo Parent, 4460-Comm Condo Parent) have $0 value and are excluded.
EXEMPT_KEYWORDS = [
    '-City', '-Religious', '-State', '-Regional', '-Charitable',
    '-Federal', '-SCC', '-Educational', '-SCLT', 'SCLT',
    'Condo Parent',  # parent parcels have $0 value
]

def is_exempt(propertyde):
    if pd.isna(propertyde) or propertyde is None:
        return False
    pd_str = str(propertyde)
    return any(kw in pd_str for kw in EXEMPT_KEYWORDS)

gdf['full_exmp'] = gdf['PROPERTYDE'].apply(is_exempt).astype(int)

# Also mark $0 total value parcels as exempt (condo parents, un-assessed parcels)
gdf.loc[gdf['TOTALVAL1'].fillna(0) == 0, 'full_exmp'] = 1

print(f"Exempt parcels: {gdf['full_exmp'].sum():,}")
print(f"Taxable parcels: {(gdf['full_exmp']==0).sum():,}")
print(f"\nExempt PROPERTYDE values:")
print(gdf[gdf['full_exmp']==1]['PROPERTYDE'].value_counts().head(20))

Exempt parcels: 2,350
Taxable parcels: 42,091

Exempt PROPERTYDE values:
PROPERTYDE
151-Vacant-SFR-City           686
155-Vacant-Religious          256
455-Comm/Indust-Religious     224
451-Comm/Indust-City          139
154-Vacant-Regional           131
152-Vacant-State              125
140-Commercial Vacant          93
100-Vacant Land                81
5580-Vacant SCLT-Other         80
354-Multifamily-Regional       67
159-Vacant Land-SCC            66
156-Vacant-Charitable          61
2260-Res Condo Parent          58
456-Comm/Indust-Charitable     42
6580-SCLT-Other                35
255-SFR-Religious              30
454-Comm/Indust-Regional       29
6560-SCLT-Charitable           22
459-Comm/Indust-SCC            21
254-SFR-Regional               21
Name: count, dtype: int64


In [5]:
# Property category mapping based on PROPERTYDE
# Taxable categories:
#   200-SingleFamily → Single Family Residential
#   100-Vacant Land, 140-Commercial Vacant → Vacant Land
#   300-Multifamily → Large Multi-Family (5+ units)
#   Multi-Family Duplex → Small Multi-Family (2-4 units)
#   400-Commercial/Industrial → Other Commercial (no use-code subcategory available)
#   420-Comm/Ind-MiscImp → Other Commercial
#   460-Commercial Condo → Office / Commercial Condo
#   220-Res-NonlivingArea → Other (garages, storage)

TAXABLE_CATEGORY_MAP = {
    '200-SingleFamily': 'Single Family Residential',
    '100-Vacant Land': 'Vacant Land',
    '140-Commercial Vacant': 'Vacant Land',
    '300-Multifamily': 'Large Multi-Family (5+ units)',
    'Multi-Family Duplex': 'Small Multi-Family (2-4 units)',
    '400-Commercial/Industrial': 'Other Commercial',
    '420-Comm/Ind-MiscImp': 'Other Commercial',
    '460-Commercial Condo': 'Office / Commercial Condo',
    '220-Res-NonlivingArea': 'Other',
}

def classify(row):
    if row['full_exmp'] == 1:
        return 'Exempt'
    pd_str = str(row.get('PROPERTYDE', '') or '')
    cat = TAXABLE_CATEGORY_MAP.get(pd_str)
    if cat:
        return cat
    # Override: $0 improvement → Vacant Land (catches $0-value commercial vacant)
    if row.get('DWELLINGVA', 0) or 0 <= 0:
        return 'Vacant Land'
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(classify, axis=1)

print("Property category distribution:")
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f"\nNull PROPERTY_CATEGORY: {gdf['PROPERTY_CATEGORY'].isna().sum()}")

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential         29615
Vacant Land                        7054
Other Commercial                   2641
Exempt                             2350
Large Multi-Family (5+ units)      1725
Small Multi-Family (2-4 units)      961
Other                                93
Office / Commercial Condo             2
Name: count, dtype: int64

Null PROPERTY_CATEGORY: 0


In [6]:
# Inspect 'Other' parcels to confirm they are not policy-relevant categories
other_mask = gdf['PROPERTY_CATEGORY'] == 'Other'
print(f"'Other' parcels: {other_mask.sum():,}")
if other_mask.any():
    print(gdf[other_mask]['PROPERTYDE'].value_counts().head(15))

'Other' parcels: 93
PROPERTYDE
220-Res-NonlivingArea    93
Name: count, dtype: int64


## Section 4 — Current Tax Model

**Virginia assessment system**: Full market value (100%) — no assessment ratio  
**City rate**: $1.22 per $100 = 12.2 mills per $1,000  
**Exemptions**: PROPERTYDE exempt-suffix parcels + $0-value parcels → excluded (current_tax = 0)  
**Revenue target**: ~$178M FY2025 (estimated from FY2026 increase data)

In [7]:
gdf['taxable_land_value'] = gdf['LANDVAL1'].fillna(0).clip(lower=0)
gdf['taxable_improvement_value'] = gdf['DWELLINGVA'].fillna(0).clip(lower=0)
gdf['taxable_total_value'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

# Consistency check against TOTALVAL1
gap = (gdf['taxable_land_value'] + gdf['taxable_improvement_value'] - gdf['TOTALVAL1'].fillna(0)).abs()
print(f"Land + Improvement vs TOTALVAL1 mismatch > $1: {(gap > 1).sum():,} parcels")

gdf['millage_rate'] = MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"\nModeled revenue:   ${current_revenue:,.0f}")
print(f"Target (FY2025 est): ${OFFICIAL_REVENUE:,.0f}")
print(f"Gap:               {gap_pct:+.2f}%")

Land + Improvement vs TOTALVAL1 mismatch > $1: 0 parcels

Modeled revenue:   $155,423,476
Target (FY2025 est): $178,000,000
Gap:               -12.68%


## Section 5 — Split-Rate Model (4:1)

In [8]:
taxable = gdf[gdf['full_exmp'] == 0].copy()
exempt = gdf[gdf['full_exmp'] == 1].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f"Land millage:        {land_millage:.4f} mills")
print(f"Improvement millage: {improvement_millage:.4f} mills")
print(f"Revenue check:       ${new_revenue:,.0f} (target: ${taxable['current_tax'].sum():,.0f})")
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title="Roanoke, VA — 4:1 Split-Rate Tax Impact")

Land millage:        30.8615 mills
Improvement millage: 7.7154 mills
Revenue check:       $155,423,476 (target: $155,423,476)


Roanoke, VA — 4:1 Split-Rate Tax Impact
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
     Single Family Residential  29615     $-2,909,189       -3.4%       $-98         $-71   -1.4%      -3.0%            11.0%            19.6%
                   Vacant Land   7054      $1,809,678       14.0%       $257         $319  152.2%     153.0%            99.6%             0.3%
              Other Commercial   2641      $3,694,899        9.3%     $1,399         $687   29.0%      19.9%            59.0%            18.5%
                        Exempt   2350              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
 Large Multi-Family (5+ units)   1725     $-2,547,691      -19.0%    $-1,477        $-148   -5.4%      -7.1%         

## Section 6 — Exploration Charts

In [9]:
taxable_summary = gdf[gdf['PROPERTY_CATEGORY'] != 'Exempt'].copy()
summary = (
    taxable_summary
    .groupby('PROPERTY_CATEGORY')['tax_change_pct']
    .median()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#d73027' if v > 0 else '#4575b4' for v in summary.values]
summary.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Roanoke, VA — Median Tax Change % by Category (4:1 Split-Rate)', fontsize=12)
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print("Saved category_preview.png")

Saved category_preview.png


## Section 7 — Census Join + Export

In [10]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [11]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/roanoke/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

  ✓ roanoke: 44,441 rows → ../../analysis/data/roanoke.csv  [model: split_rate:4.0]


Done.


## Validation Summary

| Check | Result |
|---|---|
| Revenue match | **+63.1% gap** — modeled $155M vs. official ~$95M. Cause documented below. |
| Parcel count | 44,441 (Roanoke independent city; no filter needed) |
| Census coverage | 100.0% matched to block groups |
| PNGs generated | 7 of 7 |
| SFR median change | -3.0% (slight decrease — consistent with typical LVT behavior) |
| Vacant land median change | +153.0% (very large increase — correct direction) |
| Distribution sanity | Passes: SFR decreases, vacant land and commercial parking increase |

**Revenue gap — root cause:** Roanoke's public GIS (the FeatureServer used here) labels parcels with `PROPERTYDE = '400-Commercial/Industrial'` for all taxable and many quasi-taxable commercial properties. It does NOT contain a dedicated flag for institutional non-profits that have received local tax-exemption designations under Va. Code §58.1-3607 (enacted by Roanoke City Council ordinance). The **Carilion Clinic** health system — the Roanoke Valley's largest private employer, with at least 7 parcels totaling ~$200M+ in assessed value in the top 30 parcels alone — appears as taxable in the GIS data. Carilion is a nonprofit hospital system and almost certainly exempt from local real estate tax via Va. Code §58.1-3606(A)(6) or §58.1-3607 designation. Additional institutions likely in the same category: Roanoke College, Virginia Western Community College, H.R. Foundation, Jefferson Center for the Arts, Total Action for Progress, and other charitable nonprofits. 

**The official assessment roll**, maintained by the City of Roanoke Commissioner of the Revenue, contains the complete list of exemption designations and would close this gap. This model is a conservative estimate: it overstates the taxable base and the split-rate millage rates that would result.

**Directional validity:** Despite the revenue overestimate, the RELATIVE impacts (which parcels pay more/less, and by how much) are valid. The SFR-benefits and vacant-land-penalty directions are correct regardless of the absolute scale. The model is usable for qualitative and comparative policy analysis, but should be updated with the official assessment roll before use in budget projections or official council presentations.

**Roanoke City Council tax exemption review (February 2025):** The Roanoke City Council is actively reviewing its tax exemption policy for charitable organizations — suggesting that the volume of exempt institutional property is a live policy issue in Roanoke. This context supports the LVT framing: shifting taxes from improvement to land would not affect legally exempt parcels (they pay nothing either way), but would redirect burden from improvement-intensive buildings (like Carilion's medical facilities) toward vacant land holders.